# 1. Data, point-in-time

**Question this notebook answers**
How many months of apparent macro information does the v1 panel hand a forecaster before that
information was actually published? Answer is a per-series count, summing to a panel that is
shifted by between zero and three months depending on the series.

**What v1 did**
`notebooks/viraj/data_macroeconomic.ipynb` cell 5 merges every series on a month-end key with no
release lag, and `README.md:8` describes the `GDP` column as GDP.

**Why that was wrong**
Two separate problems. First, no US macro series is available at the end of its own reference
month: the unemployment rate lands on the first Friday of the following month, CPI and industrial
production in the middle of it, and the OECD reference series two to three months later and revised
for years afterwards. A model trained on that panel is reading data that did not exist. Second, the
column named `GDP` is `EA19LORSGPORGYSAM` — the OECD Leading Indicators reference series for Euro
Area 19 GDP, year on year, interpolated monthly from quarterly. It is not US GDP, and it is the
highest-loading variable in the regime clustering.

**What this notebook does instead**
Applies the per-series `release_lag_months` recorded in `src/config_v2.py`, renames the mislabelled
column to `EA19_GDP_YoY_OECD` with its source identifier attached, and writes the aligned panel.

**Predicted effect on results — recorded before running**
Shifting the regressors should weaken their already-weak contemporaneous relationship with the
spread, so any macro-only model should get worse, not better — expect its MSE to rise. The random
walk should be untouched, since the spread itself carries no publication lag. If a macro model
improves after this change, that is a signal something is still misaligned rather than a win.

**Inputs** → **Outputs**
`data/merged_macroeconomic_credit.csv`, `data/original/*.csv` → `data/v2/panel_pit.parquet`

---
## 1.1 Load the raw merged panel

Load `data/merged_macroeconomic_credit.csv` unchanged and print shape, date range and column list. This is the v1 view, kept as the comparison point.

---
## 1.2 Confirm provenance against `data/original/`

For each series, match the merged column back to the file it came from in `data/original/` and print the source identifier. Look for the `GDP` column resolving to `EA19LORSGPORGYSAM` rather than a US series.

---
## 1.3 Publication-lag table

Print `src.data.provenance_table()`. Look for the lag applied to each series and which vintages are final-revised rather than real-time first prints.

---
## 1.4 Apply the lags

Call `src.data.apply_publication_lags`. Print how many leading rows become incomplete and what the usable sample becomes.

---
## 1.5 Rename the mislabelled column

Call `src.data.rename_to_v2`. Print the before and after column lists side by side.

---
## 1.6 Aligned versus unaligned, visually

Plot one series in both alignments on the same axes, so the shift is visible rather than asserted. Look for the lagged line trailing the original by its stated lag.

---
## 1.7 Correlation with the target, before and after

Recompute each regressor's correlation with `Credit_Spread` under both alignments. Look for correlations weakening — that is the honest cost of the correction.

---
## 1.8 Write `data/v2/panel_pit.parquet`

Call `src.data.save_panel`. Confirm the file is under `data/v2/` and that no file under `data/` itself was touched.